In [ ]:
import subprocess
import sys

packages = [
    "vllm==0.25.1",
    "transformers==5.13.0",
    "accelerate==1.14.0",
    "safetensors==0.8.0",
    "hf-xet==1.5.1",
]
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", *packages]
)
print("Frozen runtime dependencies installed.")


In [ ]:
import getpass
import os
import shutil
import zipfile
from pathlib import Path

from google.colab import drive, userdata

drive.mount("/content/drive")
try:
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = getpass.getpass("Hugging Face token: ")
if not hf_token:
    raise RuntimeError("HF_TOKEN is required for the frozen Gemma revision.")
os.environ["HF_TOKEN"] = hf_token
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["VLLM_LOGGING_LEVEL"] = "INFO"

archive = Path("/content/counterfactual_monitorability_cross_model_v2_bundle.zip")
if not archive.exists():
    raise FileNotFoundError(f"Upload {archive.name} to /content before running.")
bundle_root = Path("/content/counterfactual_monitorability_cross_model_v2_bundle")
if bundle_root.exists():
    shutil.rmtree(bundle_root)
bundle_root.mkdir(parents=True)
with zipfile.ZipFile(archive) as uploaded:
    uploaded.extractall(bundle_root)
output_root = (
    Path("/content/drive/MyDrive")
    / "Dissertation_Cross_Model_Final"
    / "counterfactual_monitorability_cross_model_v2"
)
output_root.mkdir(parents=True, exist_ok=True)
print(f"Bundle: {bundle_root}")
print(f"Drive output: {output_root}")
print(f"Live log: {output_root / 'logs' / 'orchestrator.log'}")
print(f"Progress JSON: {output_root / 'logs' / 'progress.json'}")


In [ ]:
command = [
    sys.executable,
    str(bundle_root / "src" / "run_all.py"),
    "--bundle-root",
    str(bundle_root),
    "--output-root",
    str(output_root),
]
print("Starting the frozen run. Output is mirrored to the Drive log.")
subprocess.run(command, check=True, env=os.environ.copy())


In [ ]:
import json

gate = json.loads((output_root / "OVERALL_QUALITY_GATE.json").read_text())
archive_record = json.loads((output_root / "RESULT_ARCHIVE.json").read_text())
print(json.dumps(gate, indent=2))
print()
print(f"Final result archive: {archive_record['archive_path']}")
print(f"SHA-256: {archive_record['archive_sha256']}")
